# 01 EO retardance and Vpi

Computes z-cut LiNbO3 retardance, half-wave field, and Vpi sensitivity using rigorous tensor diagonalization.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from rot_eo_model.constants import DEFAULTS
from rot_eo_model.eo import half_wave_field_V_per_m, vpi_from_gap_V, transverse_eta_matrix, eigen_indices, axis_angle_rad, full_retardance_eigenvalue

In [ ]:
Epi = half_wave_field_V_per_m(
    lambda0_m=DEFAULTS.lambda0_m, 
    length_m=DEFAULTS.crystal_length_m, 
    n_o=DEFAULTS.n_o, 
    r22_m_per_V=DEFAULTS.r22_m_per_V
)
Vpi = vpi_from_gap_V(
    DEFAULTS.electrode_gap_m, 
    lambda0_m=DEFAULTS.lambda0_m, 
    length_m=DEFAULTS.crystal_length_m, 
    n_o=DEFAULTS.n_o, 
    r22_m_per_V=DEFAULTS.r22_m_per_V
)
print(f"Small-signal Epi = {Epi:.3e} V/m = {Epi/1e5:.2f} kV/cm")
print(f"Small-signal Ideal-plate Vpi = {Vpi:.1f} Vpeak differential")

## Rigorous Numerical Diagonalization

Let's sweep the applied field and use exact numerical diagonalization of the impermeability tensor to track retardance and eigenaxis rotation.

In [ ]:
# Apply a rotating transverse field
phi_drive = np.linspace(0, 2*np.pi, 200)
E0 = Epi # drive with exact small-signal half-wave field
Ex = E0 * np.cos(phi_drive)
Ey = E0 * np.sin(phi_drive)

retardances = []
angles = []

for ex, ey in zip(Ex, Ey):
    eta = transverse_eta_matrix(ex, ey, 0.0, n_o=DEFAULTS.n_o, pockels=DEFAULTS.pockels)
    n, vecs = eigen_indices(eta)
    
    # Retardance
    dn = abs(n[1] - n[0])
    gamma = 2 * np.pi / DEFAULTS.lambda0_m * DEFAULTS.crystal_length_m * dn
    retardances.append(gamma)
    
    # Eigenaxis angle
    theta = axis_angle_rad(vecs)
    angles.append(theta)

retardances = np.array(retardances)
angles = np.array(angles)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(np.degrees(phi_drive), retardances / np.pi)
ax1.set_xlabel("Drive Phase (deg)")
ax1.set_ylabel("Retardance / $\pi$")
ax1.set_title("Retardance Ripple (exact tensor)")
ax1.grid(True)

ax2.plot(np.degrees(phi_drive), np.degrees(angles))
ax2.set_xlabel("Drive Phase (deg)")
ax2.set_ylabel("Eigenaxis Angle $\theta$ (deg)")
ax2.set_title("Eigenaxis Rotation")
ax2.grid(True)
plt.tight_layout()


## Geometry Sweep and Ambiguity Warning

If the vendor z-axis is not along the 30 mm direction, change `crystal_length_m` immediately. A 3 mm path length makes Vpi about 10x worse.

In [ ]:
Ls = np.linspace(3e-3, 50e-3, 200)
Vpis = [vpi_from_gap_V(DEFAULTS.electrode_gap_m, length_m=L) for L in Ls]
plt.figure(figsize=(6, 4))
plt.plot(Ls*1e3, Vpis, 'b-', label='Vpi')
plt.plot(3, vpi_from_gap_V(DEFAULTS.electrode_gap_m, length_m=3e-3), 'ro', label='L=3mm (Wrong Cut)')
plt.plot(30, vpi_from_gap_V(DEFAULTS.electrode_gap_m, length_m=30e-3), 'go', label='L=30mm (Desired)')
plt.xlabel("Optical interaction length L [mm]")
plt.ylabel("Vpi [Vpeak differential]")
plt.title("Vpi sensitivity to optical length")
plt.grid(True)
plt.legend()

Next: replace ideal `E=V/d` with FEM-derived field-per-volt at the beam in Notebook 02.